# Krishna Menon
## B.Tech AI I045

# Experiment 5: Wasserstein GAN (WGAN) on Fashion-MNIST

This notebook implements a **WGAN** version of your GAN pipeline from EXP4.

## What is changed vs EXP4
- Discriminator is replaced by a **Critic** (no sigmoid output)
- Uses **Wasserstein loss** instead of BCE loss
- Trains critic multiple times per generator step (`n_critic`)
- Uses **weight clipping** to enforce Lipschitz constraint

In [2]:
%pip install torch torchvision matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.8 MB 4.8 MB/s eta 0:00:24
    --------------------------------------- 2.4/113.8 MB 6.0 MB/s eta 0:00:19
   - -------------------------------------- 3.9/113.8 MB 6.6 MB/s eta 0:00:17
   -- ------------------------------------- 6.3/113.8 MB 7.2 MB/s eta 0:00:15
   --- ------------------------------------ 8.7/113.8 MB 7.6 MB/s eta 0:00:14
   --- ------------------------------------ 10.7/113.8 MB 7.9 MB/s eta 0:00:14
   ---- ----------------------------------- 12.3/113.8 MB 7.7 MB/s eta 0:00:14
   ---- ----------------------------------- 13.4/113.8 MB 7.5 MB/s eta 0:00:14
   ----- ---------------------------------- 14.4/113.8 MB 7.3 MB/s eta 0:00:14
   ----- ---------------------------------- 15.5/113.8 MB 7.0 MB/s eta 0:00:14
 

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(111)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

Torch: 2.10.0+cpu
CUDA available: False
Using device: cpu


In [ ]:
# Data pipeline (same dataset choice as EXP4)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Scale images to [-1, 1]
])

train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

batch_size = 64
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)

print("Train samples:", len(train_data))
print("Classes:", train_data.classes)

In [ ]:
# WGAN models
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.model(x)


class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.model(z)


def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)


latent_dim = 100
critic = Critic().to(device)
generator = Generator(latent_dim=latent_dim).to(device)
critic.apply(init_weights)
generator.apply(init_weights)

print("WGAN models initialized")

In [ ]:
# WGAN training setup
lr = 5e-5
num_epochs = 50
n_critic = 5          # Critic updates per generator update
clip_value = 0.01     # Weight clipping range for critic

optimizer_critic = torch.optim.RMSprop(critic.parameters(), lr=lr)
optimizer_generator = torch.optim.RMSprop(generator.parameters(), lr=lr)

critic_losses = []
generator_losses = []

print("Starting WGAN training...")

In [ ]:
for epoch in range(num_epochs):
    epoch_c_loss = 0.0
    epoch_g_loss = 0.0

    for batch_idx, (real_images, _) in enumerate(train_loader):
        real_images = real_images.view(real_images.size(0), -1).to(device)
        current_batch_size = real_images.size(0)

        # -------------------------
        # Train Critic (maximize E[C(real)] - E[C(fake)])
        # -------------------------
        for _ in range(n_critic):
            z = torch.randn(current_batch_size, latent_dim, device=device)
            fake_images = generator(z).detach()

            critic_real = critic(real_images)
            critic_fake = critic(fake_images)

            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))

            optimizer_critic.zero_grad()
            loss_critic.backward()
            optimizer_critic.step()

            # Weight clipping enforces 1-Lipschitz condition in original WGAN
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # -------------------------
        # Train Generator (minimize -E[C(fake)])
        # -------------------------
        z = torch.randn(current_batch_size, latent_dim, device=device)
        fake_images = generator(z)
        loss_generator = -torch.mean(critic(fake_images))

        optimizer_generator.zero_grad()
        loss_generator.backward()
        optimizer_generator.step()

        epoch_c_loss += loss_critic.item()
        epoch_g_loss += loss_generator.item()

    avg_c_loss = epoch_c_loss / len(train_loader)
    avg_g_loss = epoch_g_loss / len(train_loader)
    critic_losses.append(avg_c_loss)
    generator_losses.append(avg_g_loss)

    if epoch % 5 == 0:
        print(f"Epoch [{epoch}/{num_epochs}] | Critic Loss: {avg_c_loss:.4f} | Generator Loss: {avg_g_loss:.4f}")

print("WGAN training complete")

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 4))
plt.plot(critic_losses, label="Critic Loss")
plt.plot(generator_losses, label="Generator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("WGAN Training Losses")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Generate and visualize samples
num_samples = 16
generator.eval()
with torch.no_grad():
    z = torch.randn(num_samples, latent_dim, device=device)
    fake_images = generator(z).view(num_samples, 1, 28, 28)

# Unnormalize from [-1, 1] to [0, 1]
fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(fake_images[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("WGAN Generated Fashion-MNIST Samples", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Save WGAN checkpoint
save_path = "fashion_mnist_wgan_checkpoint.pth"

torch.save(
    {
        "epoch": num_epochs,
        "generator_state_dict": generator.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "optimizer_generator_state_dict": optimizer_generator.state_dict(),
        "optimizer_critic_state_dict": optimizer_critic.state_dict(),
        "latent_dim": latent_dim,
        "lr": lr,
        "n_critic": n_critic,
        "clip_value": clip_value,
    },
    save_path,
)

print(f"WGAN checkpoint saved to: {save_path}")

## Gradio Frontend Launch

In [3]:
# Launch Gradio app wired to the checkpoint generated in this project
from gradio_app import build_app

checkpoint_path = "fashion_mnist_gan_checkpoint_v1.pth"
app = build_app(checkpoint_path)

# In notebooks, set inline=False so Gradio opens in a browser tab
app.launch(inline=False, share=False)

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Defaulting to user installation because normal site-packages is not writeable
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/42.9 MB ? eta -:--:--
   - -------------------------------------- 2.1/42.9 MB 11.3 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/42.9 MB 11.3 MB/s eta 0:00:04
   ------ --------------------------------- 6.8/42.9 MB 11.3 MB/s eta 0:00:04
   -------- ------------------------------- 9.4/42.9 MB 11.4 MB/s eta 0:00:03
   ---------- ----------------------------- 11.3/42.9 MB 10.9 MB/s eta 0:00:03
   ----------- ---------------------------- 12.8/42.9 MB 10.3 MB/s e

In [2]:
%pip install gradio pillow

^C
Note: you may need to restart the kernel to use updated packages.
